In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from codeutils.data_loader import MinecraftDataLoader, visualize_sample, plot_class_distribution

#### Этап1. Загрузка и анализ данных

In [ ]:
plt.style.use('default')
%matplotlib inline

In [ ]:
DATASETS_PATH = "datasets\Minecraft_Mob_Detection"
data_loader = MinecraftDataLoader(DATASETS_PATH)

In [ ]:
train_data = data_loader.load_dataset('train')
valid_data = data_loader.load_dataset('valid')
test_data = data_loader.load_dataset('test')

In [ ]:
class_distribution = data_loader.get_class_distribution()
print("\nРАСПРЕДЕЛЕНИЕ КЛАССОВ:")
print("=" * 40)
for class_name, count in class_distribution.most_common():
    print(f"  {class_name}: {count} объектов")

plot_class_distribution(class_distribution, save_path="artifacts/metrics/class_distribution.png")
stats = data_loader.get_dataset_stats()

total_objects = sum(class_distribution.values())
max_count = max(class_distribution.values())
min_count = min(class_distribution.values())
imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')

print(f"Максимальное количество в классе: {max_count}")
print(f"Минимальное количество в классе: {min_count}")
print(f"Коэффициент дисбаланса: {imbalance_ratio:.2f}")

if imbalance_ratio > 10:
    print("Высокий дисбаланс классов!")
elif imbalance_ratio > 5:
    print("Умеренный дисбаланс классов")
else:
    print("Дисбаланс в пределах нормы")


In [ ]:
if train_data:
    sample_annotation = train_data[0]
    print(f"\nПример изображения: {sample_annotation['filename']}")
    print(f"Размер: {sample_annotation['width']}x{sample_annotation['height']}")
    print(f"Количество объектов: {len(sample_annotation['objects'])}")
    
    visualize_sample(
        sample_annotation['image_path'], 
        sample_annotation,
        save_path="artifacts/inference/sample_visualization.png"
    )

План выполнения проекта
Использование аугментации для балансировки классов. Использование современных детекторов объектов (FCOS и YOLO). Fine-tuning предобученных моделей на данных Minecraft. Обработка возможного дисбаланса классов.
Возможная окклюзия объектов в Minecraft

#### Этап 2. Настройка конфигурации моделей

In [ ]:
import torch
import torchvision

from codeutils.fcos_inference_pytorch import FCOSInferencePyTorch

In [ ]:
%matplotlib inline
plt.rcParams['figure.figsize'] = [15, 10]
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Используемое устройство: {device}")

In [ ]:
print("=== ПРОВЕРКА ДОСТУПНЫХ МОДЕЛЕЙ ===")
detection_models = [attr for attr in dir(torchvision.models.detection) 
                       if not attr.startswith('_') and not attr[0].islower()]
    
for i, model_name in enumerate(detection_models, 1):
    print(f"  {i:2d}. {model_name}")

if 'FCOS_ResNet50_FPN_Weights' in detection_models:
    print("✓ FCOS модель доступна")
else:
    print("FCOS модель недоступна в этой версии torchvision")

In [ ]:
print("\n=== ТЕСТИРОВАНИЕ FCOS ИНФЕРЕНСА (PyTorch) ===")

from codeutils.fcos_inference_pytorch import FCOSInferencePyTorch

try:
    # Инициализация тестера
    fcos_tester = FCOSInferencePyTorch(
        model_path=None,  # Используем pretrained модель COCO
        device=device
    )

    # Тестирование инференса
    fcos_result = fcos_tester.test_inference(save_path='artifacts/inference/test_pretrained.jpg')
    print(f"FCOS инференс завершен: {fcos_result}")
    
except Exception as e:
    print(f"Ошибка при инициализации FCOS: {e}")

In [ ]:
print("\n=== ТЕСТИРОВАНИЕ YOLO ИНФЕРЕНСА ===")

try:
    from codeutils.yolo_inference import YOLOInferenceTester

    # Инициализация YOLO тестера
    yolo_tester = YOLOInferenceTester(model_type='yolov8s')

    # Тестирование инференса
    yolo_result = yolo_tester.test_inference(save_dir='artifacts/inference/yolo_val/')
    print(f"YOLO инференс завершен: {yolo_result}")
    
except Exception as e:
    print(f"Ошибка при YOLO инференсе: {e}")

#### Этап 3. Обучение моделей FCOS и YOLO

In [ ]:
import numpy as np

In [ ]:
from codeutils.minecraft_dataset import MinecraftDataset, collate_fn
from configs.fcos_minecraft import FCOSMinecraftConfig

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Используемое устройство: {device}")

In [ ]:
print("\n=== ОБУЧЕНИЕ FCOS МОДЕЛИ ===")

from codeutils.working_fcos_trainer import WorkingFCOSTrainer
from codeutils.minecraft_dataset import MinecraftDataset, collate_fn

# Загрузка датасетов с конфигом
print("Загрузка датасетов с маппингом классов...")
train_dataset = MinecraftDataset(
    'datasets\Minecraft_Mob_Detection',
    transform=FCOSMinecraftConfig.train_transforms,
    split='train',
    config=FCOSMinecraftConfig
)

val_dataset = MinecraftDataset(
    'datasets\Minecraft_Mob_Detection', 
    transform=FCOSMinecraftConfig.val_transforms,
    split='valid',
    config=FCOSMinecraftConfig
)

# DataLoaders
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=FCOSMinecraftConfig.batch_size,
    shuffle=True,
    num_workers=FCOSMinecraftConfig.num_workers,
    collate_fn=collate_fn
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=FCOSMinecraftConfig.batch_size,
    shuffle=False,
    num_workers=FCOSMinecraftConfig.num_workers,
    collate_fn=collate_fn
)

print(f"Тренировочный датасет: {len(train_dataset)} изображений")
print(f"Валидационный датасет: {len(val_dataset)} изображений")

# Создание модели
print("Создание FCOS модели...")
model = FCOSMinecraftConfig.get_model(pretrained=True)
model.to(device)

print("✓ Модель успешно создана!")
print(f"Количество классов в модели: {FCOSMinecraftConfig.num_classes}")
print(f"Маппинг наших классов: {FCOSMinecraftConfig.class_mapping}")

# Проверка модели
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Всего параметров: {total_params:,}")
print(f"Обучаемых параметров: {trainable_params:,}")

# Запуск обучения
trainer = WorkingFCOSTrainer(model, train_loader, val_loader, FCOSMinecraftConfig, device=device)
fcos_results = trainer.train()

print("Обучение FCOS завершено!")

